# 4D SfM Pipeline

Processes one new day at a time through the full 4D SfM workflow. All logic
lives in `cntp.pipeline_4dsfm.run_4dsfm_day`; this notebook is just config +
one call.

Steps run by `run_4dsfm_day` (each skips if its output already exists):

1. Multi-temporal bundle adjustment — reference cameras pinned tight (0.001 m),
   new-day cameras loose; new-day IOP floating, ref IOP fixed per-day.
2. Single-day re-run with **fixed** IOP from Step 1.
3. ASP three-stage ICP (`point-to-plane` → `similarity-p2p` → stable terrain only).
4. Apply ASP transform to the single-day camera EOPs.
5. _(no-op — verification only)_
6. Rebuild the cloud from the corrected Metashape chunk transform (same session
   as the matrix fix, so it isn't lost on save/reload).
7. Append the validated day to the reference registry.

Step 3b runs M3C2 on the coreg cloud; Step 6b validates that the rebuilt cloud
matches the coreg cloud (median ≈ 0 m → transform propagated correctly).


In [1]:
%load_ext autoreload
%autoreload 2

import os
os.environ["AGISOFT_LICENSE_PATH"] = "/home/asus/.config/Agisoft/license.lic"

from pathlib import Path

import Metashape  # noqa: F401  — must import after AGISOFT_LICENSE_PATH is set
from cntp.pipeline_4dsfm import run_4dsfm_day


## Configuration

Set all paths and parameters here. Defaults match the values that produced the
validated 2023-12-15 run; tweak per-day if needed.


In [2]:
# ── Paths ────────────────────────────────────────────────────────────────
base_dir     = Path("/mnt/g/2023_11_Nepal/2023_Changri")
tlcam_dir    = base_dir / "TLCAM"
output_dir   = base_dir

ref_cloud    = base_dir / "Ref_PC" / "Reference_UAV_TLC_PCS.laz"
glacier_mask = base_dir / "glaciermask_new" / "glacier_mask_pcs.shp"
registry_csv = output_dir / "output_new" / "reference_registry.csv"

# ── Date to process ──────────────────────────────────────────────────────
new_date = "2023-12-15"

# ── Pipeline parameters (defaults shown — override as needed) ────────────
params = dict(
    match_downscale = 0,
    depth_downscale = 2,
    loc_acc_new     = (0.5, 0.5, 0.5),
    rot_acc_new     = (5.0, 5.0, 5.0),
    ref_downsample  = 0.4,
    tba_downsample  = 1.0,
    p2p_max_disp    = 10.0,
    sp2p_max_disp   =  5.0,
    m_sp2p_max_disp =  2.0,
    use_ecef        = True,
    overwrite       = False,   # True forces full recompute
    verbose         = True,    # print pc_align stdout
)


## Run

`overwrite=False` means each step skips itself if its key output exists on
disk — useful for resuming after a crash. Set `overwrite=True` above for a
fresh run.


In [3]:
result = run_4dsfm_day(
    new_date     = new_date,
    tlcam_dir    = tlcam_dir,
    ref_cloud    = ref_cloud,
    glacier_mask = glacier_mask,
    registry_csv = registry_csv,
    output_dir   = output_dir,
    **params,
)

print()
print(f"Coreg M3C2  : before {result['coreg_med_before']:+.4f} m  →  after {result['coreg_med_after']:+.4f} m")
print(f"Validation  : median {result['validation_med']:+.4f} m  std {result['validation_std']:.4f} m")
print(f"Validated   : {result['validated_laz']}")


[Step 1] Skipping — 2023-12-15_cameras_4DSfM.csv exists
[Step 2] Skipping — 2023-12-15_cloud.las exists
  Saving downsampled reference (40%) → Reference_UAV_TLC_PCS_ds0.40.las
  Stable reference → /mnt/g/2023_11_Nepal/2023_Changri/output_new/_ref_cache/Reference_UAV_TLC_PCS_ds0.40_stable.las  (1,003,575 pts)

[Step 3] ASP 3-stage ICP — 2023-12-15
  Stage 1 — point-to-plane ICP (max_displacement=10.0 m)
pc_align --threads 16 --max-displacement 10.0 --alignment-method point-to-plane -o /mnt/g/2023_11_Nepal/2023_Changri/output_new/2023-12-15/coreg/stage1/run /mnt/g/2023_11_Nepal/2023_Changri/output_new/2023-12-15/coreg/stage1/ecef/Reference_UAV_TLC_PCS_ds0.40_ecef.las /mnt/g/2023_11_Nepal/2023_Changri/output_new/2023-12-15/coreg/stage1/ecef/2023-12-15_cloud_ecef.las
	--> Setting number of processing threads to: 16
Writing log: /mnt/g/2023_11_Nepal/2023_Changri/output_new/2023-12-15/coreg/stage1/run-log-pc_align-05-19-1747-17220.txt
No datum specified. Will write output CSV files in the x,

Can't load OpenCL library
failed to enable cpu vulkan support (/home/asus/miniconda3/envs/cntp/bin/lib dir not exists)


Found 1 GPUs in 1.5974 sec (CUDA: 1.29184 sec, OpenCL: 0.001354 sec, Vulkan: 0.30389 sec)
Using device: NVIDIA GeForce RTX 3050 Ti Laptop GPU, 20 compute units, free memory: 3277/4095 MB, compute capability 8.6
  driver/runtime CUDA: 12070/10010
  max work group size 1024
  max work item sizes [1024, 1024, 64]
  got device properties in 0.000614 sec, free memory in 0.726077 sec
group 1/1: cameras images prepared in 8.00458 s
group 1/1: 44 x frame
group 1/1: 44 x uint8
group 1/1: expected peak VRAM usage: 945 MB (404 MB max alloc, 6512x7326 mipmap texture, 14 max neighbors)
Found 1 GPUs in 0.00076 sec (CUDA: 1.8e-05 sec, OpenCL: 0.000459 sec, Vulkan: 0.000255 sec)
Using device: NVIDIA GeForce RTX 3050 Ti Laptop GPU, 20 compute units, free memory: 3277/4095 MB, compute capability 8.6
  driver/runtime CUDA: 12070/10010
  max work group size 1024
  max work item sizes [1024, 1024, 64]
Using device 'NVIDIA GeForce RTX 3050 Ti Laptop GPU' in concurrent. (2 times)
[GPU 1] group 1/1: estimatin

Can't load OpenCL library


[GPU 1] Camera 0 samples after final filtering: 16% (0.884293 avg inliers) = 100% - 26% (not matched) - 31% (bad matched) - 5% (no neighbors) - 5% (no cost neighbors) - 8% (inconsistent normal) - 0% (estimated bad angle) - 0% (found bad angle) - 8% (speckles filtering)
[GPU 1] Camera 0 tile #1/4: level #6/6 (x2 downscale: 1664x1280, image blowup: 3328x2560) done in 0.983706 s = 47% propagation + 38% refinement + 7% filtering + 0% smoothing
Peak VRAM usage updated: Camera 0 (3 neihbs): 157 MB = 48 MB gpu_neighbImages (31%) + 19 MB gpu_mipmapNeighbImage (12%) + 12 MB gpu_tmp_normal (8%) + 12 MB gpu_tmp_hypo_ni_cost (8%) + 8 MB gpu_refImage (5%) + 8 MB gpu_depth_map (5%) + 8 MB gpu_cost_map (5%) + 8 MB gpu_coarse_depth_map_radius (5%) + 8 MB gpu_coarse_depth_map (5%) + 6 MB gpu_normal_map (4%)
[GPU 2] Camera 1 samples after final filtering: 14% (0.942482 avg inliers) = 100% - 29% (not matched) - 30% (bad matched) - 5% (no neighbors) - 8% (no cost neighbors) - 8% (inconsistent normal) - 0%

Can't load OpenCL library


Camera 0 (3 neighbs) level #1/3 filtering: 1% good (1% of speckles) + 4% norm (99% of speckles) - 0% speckles + 4% bad + 91% empty (14% inliers support + 12% inliers intersects + 8% inliers doesn't reach + 13% inliers no depth + 28% outliers support + 26% outliers intersects + 16% outliers doesn't reach + 2% inliers occludes + 5% outliers occludes)
Camera 0 (3 neighbs) level #2/3 filtering: 2% good (0% of speckles) + 9% norm (100% of speckles) - 0% speckles + 8% bad + 81% empty (15% inliers support + 11% inliers intersects + 7% inliers doesn't reach + 12% inliers no depth + 31% outliers support + 23% outliers intersects + 15% outliers doesn't reach + 7% inliers occludes + 16% outliers occludes)
Camera 0 (3 neighbs) level #3/3 filtering: 3% good (1% of speckles) + 14% norm (99% of speckles) - 1% speckles + 10% bad + 73% empty (19% inliers support + 12% inliers intersects + 7% inliers doesn't reach + 10% inliers no depth + 36% outliers support + 17% outliers intersects + 11% outliers doe